# Hospedando Servidor MCP no Amazon Bedrock AgentCore Runtime - Autenticação de Entrada AWS IAM

## Visão Geral

Neste tutorial aprenderemos como hospedar servidores MCP (Model Context Protocol) no Amazon Bedrock AgentCore Runtime. Usaremos o SDK Python do Amazon Bedrock AgentCore para encapsular ferramentas MCP como um servidor MCP compatível com o Amazon Bedrock AgentCore.

O SDK Python do Amazon Bedrock AgentCore cuida dos detalhes de implementação do servidor MCP para que você possa se concentrar na funcionalidade principal das suas ferramentas. Ele transforma seu código nos contratos padronizados do protocolo MCP do AgentCore para comunicação direta.

Embora a especificação do [protocolo MCP](https://modelcontextprotocol.io/docs/getting-started/intro) tradicionalmente exija tokens OAuth para autenticação, o AgentCore runtime permite a capacidade de configurar credenciais AWS IAM para requisições de entrada aos seus servidores MCP, atendendo a um requisito empresarial crucial.

### Detalhes do Tutorial

| Informação          | Detalhes                                                  |
|:--------------------|:----------------------------------------------------------|
| Tipo de tutorial    | Hospedagem de Ferramentas                                 |
| Tipo de ferramenta  | Servidor MCP                                              |
| Componentes         | Hospedagem de servidor MCP no AgentCore Runtime           |
| Vertical            | Cross-vertical                                            |
| Complexidade        | Fácil                                                     |
| SDK usado           | SDK Python do Amazon BedrockAgentCore e MCP               |

### Arquitetura do Tutorial

Neste tutorial descreveremos como implantar um servidor MCP no AgentCore runtime.

Para fins de demonstração, usaremos um servidor MCP simples com 3 ferramentas: `add_numbers`, `multiply_numbers` e `greet_user`

<div style="text-align:left">
    <img src="images/hosting_mcp_server.png" width="60%"/>
</div>

### Recursos Principais do Tutorial

* Criação de servidores MCP com ferramentas personalizadas
* Teste de servidores MCP localmente
* Hospedagem de servidores MCP no Amazon Bedrock AgentCore Runtime
* Invocação de servidores MCP implantados com autenticação


## Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10+
* Credenciais AWS configuradas
* SDK do Amazon Bedrock AgentCore
* Biblioteca MCP (Model Context Protocol)
* Daemon Docker em execução

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_control_client = boto_session.client("bedrock-agentcore-control", region_name=region)
ssm_client = boto_session.client('ssm', region_name=region)

tool_name = "mcp_server_iam"

## Entendendo MCP (Model Context Protocol)

MCP é um protocolo que permite que modelos de IA acessem com segurança dados externos e ferramentas. Conceitos principais:

* **Ferramentas**: Funções que a IA pode chamar para executar ações
* **Streamable HTTP**: Protocolo de transporte usado pelo AgentCore Runtime
* **Isolamento de Sessão**: Cada cliente recebe sessões isoladas via cabeçalho `Mcp-Session-Id`
* **Operação Stateless**: Servidores devem suportar operação stateless para escalabilidade

O AgentCore Runtime espera que servidores MCP sejam hospedados em `0.0.0.0:8000/mcp` como o caminho padrão.

### Estrutura do Projeto

Vamos configurar nosso projeto com a estrutura adequada:

```
mcp_server_project/
├── mcp_server.py              # Código principal do servidor MCP
├── mcp_client.py          # Cliente de teste local
├── mcp_client_remote.py   # Cliente de teste remoto
├── requirements.txt          # Dependências
└── __init__.py              # Marcador de pacote Python
```

## Criando Servidor MCP

Agora vamos criar nosso servidor MCP com três ferramentas simples. O servidor usa FastMCP com `stateless_http=True`, que é necessário para compatibilidade com o AgentCore Runtime.

In [ ]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP
from starlette.responses import JSONResponse

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together"""
    return a + b

@mcp.tool()
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers together"""
    return a * b

@mcp.tool()
def greet_user(name: str) -> str:
    """Greet a user by name"""
    return f"Hello, {name}! Nice to meet you."

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### O que Este Código Faz

* **FastMCP**: Cria um servidor MCP que pode hospedar suas ferramentas
* **@mcp.tool()**: Decorador que transforma suas funções Python em ferramentas MCP
* **stateless_http=True**: Necessário para compatibilidade com o AgentCore Runtime
* **Ferramentas**: Três ferramentas simples demonstrando diferentes tipos de operações

## Criando Cliente de Teste Local

Antes de implantar no AgentCore Runtime, vamos criar um cliente para testar nosso servidor MCP localmente:

In [ ]:
%%writefile mcp_client.py
import asyncio

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    mcp_url = "http://localhost:8000/mcp"
    headers = {}

    async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tool_result = await session.list_tools()
            print("Available tools:")
            for tool in tool_result.tools:
                print(f"  - {tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(main())

 ### Testando Localmente

Para testar seu servidor MCP localmente:

1. **Terminal 1**: Inicie o servidor MCP
   ```bash
   python mcp_server.py
   ```
   
2. **Terminal 2**: Execute o cliente de teste
   ```bash
   python mcp_client.py
   ```

Você deve ver suas três ferramentas listadas na saída.

## Configurando Implantação do AgentCore Runtime

Em seguida, usaremos nosso kit inicial para configurar a implantação do AgentCore Runtime com um entrypoint, a role de execução que acabamos de criar e um arquivo de requisitos. Também configuraremos o kit inicial para criar automaticamente o repositório Amazon ECR no lançamento.

Durante a etapa de configuração, seu arquivo docker será gerado com base no código da sua aplicação

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [ ]:
import os
from bedrock_agentcore_starter_toolkit import Runtime

print(f"Using AWS region: {region}")

required_files = ["mcp_server.py", "requirements.txt"]
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")

agentcore_runtime = Runtime()

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    protocol="MCP",
    agent_name=tool_name,
)
print("Configuration completed ✓")

## Lançando Servidor MCP no AgentCore Runtime

Agora que temos um arquivo docker, vamos lançar o servidor MCP no AgentCore Runtime. Isso criará o repositório Amazon ECR e o AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [ ]:
print("Launching MCP server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch()
print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

In [ ]:

agent_arn_response = ssm_client.put_parameter(
    Name='/mcp_server/runtime_iam/agent_arn',
    Value=launch_result.agent_arn,
    Type='String',
    Description='Agent ARN for MCP server with inbound auth',
    Overwrite=True
)
print("✓ Agent ARN stored in Parameter Store")

print("\nConfiguration stored successfully!")
print(f"Agent ARN: {launch_result.agent_arn}")

### Demonstração do Ciclo de Vida da Sessão: Parando uma Sessão

Agora que o runtime está implantado, vamos demonstrar como parar uma sessão. Invocaremos o
servidor MCP com um ID de sessão personalizado e depois o pararemos para liberar os recursos do microVM enquanto
mantemos o runtime ativo para novas sessões.

In [ ]:
import uuid
import asyncio
from streamable_http_sigv4 import streamablehttp_client_with_sigv4
from mcp import ClientSession

# Generate custom session ID
demo1_session_id = str(uuid.uuid4())
print(f"📝 Demo 1 - Generated mcpSessionId: {demo1_session_id}")

# Prepare MCP URL
encoded_arn = launch_result.agent_arn.replace(':', '%3A').replace('/', '%2F')
mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

credentials = boto_session.get_credentials()
headers = {"Mcp-Session-Id": demo1_session_id}

# Invoke with custom session ID
async def test_session():
    async with streamablehttp_client_with_sigv4(
        url=mcp_url, credentials=credentials, service="bedrock-agentcore",
        region=region, headers=headers
    ) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print(f"✅ Session active with {len(tools.tools)} tools")

await test_session()

# Stop the session
print(f"\n🛑 Stopping session '{demo1_session_id}'...")
agentcore_client = boto_session.client('bedrock-agentcore', region_name=region)
response = agentcore_client.stop_runtime_session(
    agentRuntimeArn=launch_result.agent_arn,
    runtimeSessionId=demo1_session_id,
    qualifier='DEFAULT'
)
print(f"✅ Session stopped (HTTP {response['ResponseMetadata']['HTTPStatusCode']})")
print(f"   Request ID: {response['ResponseMetadata']['RequestId']}")
print(f"💡 Runtime remains alive for new sessions")

# Note: You may see 'Session termination failed: 404' in the logs above.
# This is expected - it's the MCP client trying to auto-cleanup after we already stopped the session.
# The important part is the 'HTTP 200' response from our explicit stop_runtime_session call.

## Criando Cliente de Teste Remoto

Agora vamos criar um cliente para testar nosso servidor MCP implantado. Este cliente recuperará as credenciais necessárias da AWS e se conectará ao servidor implantado:

In [ ]:
%%writefile mcp_client_remote.py       
import asyncio
import sys
import logging
import boto3
from boto3.session import Session
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from streamable_http_sigv4 import streamablehttp_client_with_sigv4


logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


def create_streamable_http_transport_sigv4(
    mcp_url: str, service_name: str, region: str
):
    """
    Create a streamable HTTP transport with AWS SigV4 authentication.

    This function creates an MCP client transport that uses AWS Signature Version 4 (SigV4)
    to authenticate requests. This is necessary because standard MCP clients don't natively
    support AWS IAM authentication, and this bridges that gap.

    Args:
        mcp_url (str): The URL of the MCP gateway endpoint
        service_name (str): The AWS service name for SigV4 signing (typically "bedrock-agentcore")
        region (str): The AWS region where the gateway is deployed

    Returns:
        StreamableHTTPTransportWithSigV4: A transport instance configured for SigV4 auth

    Example:
        >>> transport = create_streamable_http_transport_sigv4(
        ...     mcp_url=".../mcp",
        ...     service_name="bedrock-agentcore",
        ...     region="us-west-2"
        ... )
    """
    # Get AWS credentials from the current boto3 session
    # These credentials will be used to sign requests with SigV4
    session = boto3.Session()
    credentials = session.get_credentials()

    # Create and return the custom transport with SigV4 signing capability
    return streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=credentials,
        service=service_name,
        region=region,
    )


def get_full_tools_list(client):
    """
    Retrieve the complete list of tools from an MCP client, handling pagination.

    MCP servers may return tools in paginated responses. This function handles the
    pagination automatically and returns all available tools in a single list.

    Args:
        client: An MCP client instance (from strands.tools.mcp.mcp_client.MCPClient)

    Returns:
        list: A complete list of all tools available from the MCP server

    Example:
        >>> mcp_client = MCPClient(lambda: create_transport())
        >>> all_tools = get_full_tools_list(mcp_client)
        >>> print(f"Found {len(all_tools)} tools")
    """
    more_tools = True
    tools = []
    pagination_token = None

    # Loop until we've fetched all pages
    while more_tools:
        tmp_tools = client.list_tools_sync(pagination_token=pagination_token)

        tools.extend(tmp_tools)

        # Check if there are more pages to fetch
        if tmp_tools.pagination_token is None:
            # No more pages - we're done
            more_tools = False
        else:
            # More pages exist - prepare to fetch the next one
            more_tools = True
            pagination_token = tmp_tools.pagination_token

    return tools


async def main():
    boto_session = Session()
    region = boto_session.region_name
    print(f"Using AWS region: {region}")

    ssm_client = boto3.client("ssm", region_name=region)

    agent_arn_response = ssm_client.get_parameter(
        Name="/mcp_server/runtime_iam/agent_arn"
    )
    agent_arn = agent_arn_response["Parameter"]["Value"]
    print(f"Retrieved Agent ARN: {agent_arn}")

    if not agent_arn:
        print("❌ Error: AGENT_ARN not found")
        sys.exit(1)

    encoded_arn = agent_arn.replace(":", "%3A").replace("/", "%2F")
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

    try:
        async with create_streamable_http_transport_sigv4(
            mcp_url=mcp_url, service_name="bedrock-agentcore", region=region
        ) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")

                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()

                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}")
                    print(f"   Description: {tool.description}")
                    if hasattr(tool, "inputSchema") and tool.inputSchema:
                        properties = tool.inputSchema.get("properties", {})
                        if properties:
                            print(f"   Parameters: {list(properties.keys())}")
                    print()

                print(f"✅ Successfully connected to MCP server!")
                print(f"Found {len(tool_result.tools)} tools available.")

    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        import traceback

        print("\n🔍 Full error traceback:")
        traceback.print_exc()
        sys.exit(1)


if __name__ == "__main__":
    asyncio.run(main())


## Testando Seu Servidor MCP Implantado

Vamos testar nosso servidor MCP implantado usando o cliente remoto:

In [ ]:
print("Testing deployed MCP server...")
print("=" * 50)
!python mcp_client_remote.py

### Invocando Ferramentas MCP Remotamente

Agora vamos criar um cliente aprimorado que não apenas lista ferramentas, mas também as invoca para demonstrar a funcionalidade completa do MCP:

In [ ]:
%%writefile invoke_mcp_tools.py
import asyncio
import sys
import os
import logging
import boto3
import uuid
from boto3.session import Session
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from streamable_http_sigv4 import streamablehttp_client_with_sigv4

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


def create_streamable_http_transport_sigv4(
    mcp_url: str, service_name: str, region: str
):
    """Create a streamable HTTP transport with AWS SigV4 authentication."""
    session = boto3.Session()
    credentials = session.get_credentials()
    return streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=credentials,
        service=service_name,
        region=region,
    )


async def main():
    boto_session = Session()
    region = boto_session.region_name
    print(f"Using AWS region: {region}")

    ssm_client = boto3.client("ssm", region_name=region)

    agent_arn_response = ssm_client.get_parameter(
        Name="/mcp_server/runtime_iam/agent_arn"
    )
    agent_arn = agent_arn_response["Parameter"]["Value"]
    print(f"Retrieved Agent ARN: {agent_arn}")

    if not agent_arn:
        print("❌ Error: AGENT_ARN not found")
        sys.exit(1)

    encoded_arn = agent_arn.replace(":", "%3A").replace("/", "%2F")
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

    # Generate custom session ID
    mcp_session_id = str(uuid.uuid4())
    print(f"\n📝 Generated custom mcpSessionId: {mcp_session_id}")
    
    # Get credentials for SigV4
    credentials = boto_session.get_credentials()
    
    # Pass custom session ID as header
    headers = {"Mcp-Session-Id": mcp_session_id}

    try:
        async with streamablehttp_client_with_sigv4(
                url=mcp_url,
                credentials=credentials,
                service="bedrock-agentcore",
                region=region,
                headers=headers
        ) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")

                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()

                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}: {tool.description}")

                print("\n🧪 Testing MCP Tools:")
                print("=" * 50)

                try:
                    print("\n➕ Testing add_numbers(5, 3)...")
                    add_result = await session.call_tool(
                        name="add_numbers", arguments={"a": 5, "b": 3}
                    )
                    print(f"   Result: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                try:
                    print("\n✖️  Testing multiply_numbers(4, 7)...")
                    multiply_result = await session.call_tool(
                        name="multiply_numbers", arguments={"a": 4, "b": 7}
                    )
                    print(f"   Result: {multiply_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                try:
                    print("\n👋 Testing greet_user('Alice')...")
                    greet_result = await session.call_tool(
                        name="greet_user", arguments={"name": "Alice"}
                    )
                    print(f"   Result: {greet_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                print("\n✅ MCP tool testing completed!")
        
        # Demonstrate stopping the session
        print(f"\n🛑 Stopping session '{mcp_session_id}'...")
        agentcore_client = boto3.client('bedrock-agentcore', region_name=region)
        response = agentcore_client.stop_runtime_session(
            agentRuntimeArn=agent_arn,
            runtimeSessionId=mcp_session_id,
            qualifier='DEFAULT'
        )
        print(f"✅ Session stopped (HTTP {response['ResponseMetadata']['HTTPStatusCode']})")
        print(f"   Request ID: {response['ResponseMetadata']['RequestId']}")
        print(f"   MicroVM resources released")
        print(f"💡 Runtime remains alive for new sessions")

    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        import traceback

        print("\n🔍 Full error traceback:")
        traceback.print_exc()
        sys.exit(1)


if __name__ == "__main__":
    asyncio.run(main())


## Testar Invocação de Ferramentas

Vamos testar nossas ferramentas MCP invocando-as de fato:

In [ ]:
print("Testing MCP tool invocation...")
print("=" * 50)
!python invoke_mcp_tools.py

### Demonstração do Ciclo de Vida da Sessão: Parando Sessão Anterior

Antes de passar para a abordagem Boto3, vamos parar a sessão do teste do cliente MCP anterior.
Isso demonstra que você pode parar sessões em qualquer ponto do seu fluxo de trabalho.

In [ ]:
# The previous mcp_client_remote.py test created a session
# We can stop it here to demonstrate session management between different test approaches
print("💡 Note: The previous MCP client test created a session that we could stop here.")
print("   In production, track session IDs from your invocations and stop them when done.")
print("   For this demo, we'll create and stop a new session to show the pattern.")

# Create and immediately stop a session to demonstrate
demo2_session_id = str(uuid.uuid4())
print(f"\n📝 Demo 2 - Generated mcpSessionId: {demo2_session_id}")

async def quick_session():
    async with streamablehttp_client_with_sigv4(
        url=mcp_url, credentials=credentials, service="bedrock-agentcore",
        region=region, headers={"Mcp-Session-Id": demo2_session_id}
    ) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            print(f"✅ Session {demo2_session_id} created")

await quick_session()

# Stop it
print(f"🛑 Stopping session '{demo2_session_id}'...")
response = agentcore_client.stop_runtime_session(
    agentRuntimeArn=launch_result.agent_arn,
    runtimeSessionId=demo2_session_id,
    qualifier='DEFAULT'
)
print(f"✅ Session stopped (HTTP {response['ResponseMetadata']['HTTPStatusCode']})")
print(f"   Request ID: {response['ResponseMetadata']['RequestId']}")
print(f"   Ready for Boto3 approach test")

## Testando servidor MCP remoto - Abordagem Boto3 

Esta seção demonstra uma abordagem alternativa para testar o servidor MCP implantado usando a API `invoke_agent_runtime` do SDK Boto3. O SDK gerencia a assinatura de requisições AWS SigV4 automaticamente, simplificando a autenticação IAM para invocações de runtime

### Criar Cliente de Teste Remoto - Boto3

Vamos criar um cliente para testar nosso servidor MCP implantado usando a API Boto3: 

In [ ]:
%%writefile mcp_client_remote_boto3.py   

import boto3
import json
import traceback
from boto3.session import Session
from botocore.exceptions import ClientError

boto_session = Session()
region = boto_session.region_name
print(f"Using AWS region: {region}")

# Initialize the Bedrock AgentCore and SSM client
client = boto3.client('bedrock-agentcore', region_name=region)
ssm_client = boto3.client("ssm", region_name=region)


agent_arn_response = ssm_client.get_parameter(
        Name="/mcp_server/runtime_iam/agent_arn"
)

runtime_arn = agent_arn_response["Parameter"]["Value"]

print(f"Retrieved Agent ARN: {runtime_arn}")

if not runtime_arn:
        print("❌ Error: AGENT_ARN not found")
        sys.exit(1)
        
def call_mcp(method, params=None):
    """
    Call an MCP method on the agent runtime.
    
    Args:
        method: The MCP method to call (e.g., 'tools/list', 'tools/call')
        params: Optional parameters for the method
    
    Returns:
        The result from the MCP response
    """
    if params is None:
        params = {}

    payload = json.dumps({
        "jsonrpc": "2.0",
        "id": 1,
        "method": method,
        "params": params
    }).encode()

    try:
        response = client.invoke_agent_runtime(
            agentRuntimeArn=runtime_arn,
            payload=payload,
            qualifier='DEFAULT',
            contentType='application/json',
            accept='application/json, text/event-stream'
        )

        raw = response['response'].read().decode()
        json_data = json.loads(raw[raw.find('{'):])
        return json_data['result']

    except ClientError as e:
        print(f"\n{'=' * 60}")
        print("Error Response:")
        print(json.dumps(e.response, indent=2, default=str))
        print(f"{'=' * 60}\n")
        raise


def main():

    try:
        # List available tools
        print("📋 Available MCP Tools:")
        print("=" * 50)
        
        tools_result = call_mcp("tools/list")
        tools = tools_result['tools']
        
        for tool in tools:
            params = list(tool.get('inputSchema', {}).get('properties', {}).keys())
            print(f"🔧 {tool['name']}")
            print(f"   Description: {tool['description']}")
            print(f"   Parameters: {params}")
            print()
        
        print(f"✅ Successfully connected to MCP server!")
        print(f"Found {len(tools)} tools available.")

    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        import traceback

        print("\n🔍 Full error traceback:")
        traceback.print_exc()
        sys.exit(1)

if __name__ == "__main__":
    main()


### Testando Seu Servidor MCP Implantado

Vamos testar nosso servidor MCP implantado usando o cliente remoto:

In [ ]:
print("Testing deployed MCP server...")
print("=" * 50)
!python mcp_client_remote_boto3.py

### Invocar Ferramentas MCP - Boto3

Agora vamos usar o SDK boto3 para criar um cliente que não apenas lista ferramentas, mas também as invoca:

In [ ]:
%%writefile invoke_mcp_tools_boto3.py

import boto3
import json
import logging
from boto3.session import Session
from botocore.exceptions import ClientError

boto_session = Session()
region = boto_session.region_name
client = boto3.client('bedrock-agentcore', region_name=region)

ssm_client = boto3.client("ssm", region_name=region)
agent_arn_response = ssm_client.get_parameter(Name="/mcp_server/runtime_iam/agent_arn")
runtime_arn = agent_arn_response["Parameter"]["Value"]

def call_mcp(method, params=None):
    if params is None:
        params = {}
    payload = json.dumps({
        "jsonrpc": "2.0",
        "id": 1,
        "method": method,
        "params": params
    }).encode()
    try:
        response = client.invoke_agent_runtime(
            agentRuntimeArn=runtime_arn,
            payload=payload,
            qualifier='DEFAULT',
            contentType='application/json',
            accept='application/json, text/event-stream'
        )
        raw = response['response'].read().decode()
        json_data = json.loads(raw[raw.find('{'):])
        return json_data['result']
    except ClientError as e:
        print(f"❌ Error: {e}")
        raise

def main():
    
    print(f"Using AWS region: {region}")
    print(f"Retrieved Agent ARN: {runtime_arn}")


    print("\n🔄 Listing available tools...")
    try: 
        tools_result = call_mcp("tools/list")

        print("\n📋 Available MCP Tools:")
        print("=" * 50)
        for tool in tools_result['tools']:
            print(f"🔧 {tool['name']}: {tool['description']}")

        print("\n🧪 Testing MCP Tools:")
        print("=" * 50)

        print("\n➕ Testing add_numbers(5, 3)...")
        add_result = call_mcp("tools/call", {"name": "add_numbers", "arguments": {"a": 5, "b": 3}})
        print(f"   Result: {add_result['structuredContent']}")

        print("\n✖️  Testing multiply_numbers(4, 7)...")
        multiply_result = call_mcp("tools/call", {"name": "multiply_numbers", "arguments": {"a": 4, "b": 7}})
        print(f"   Result: {multiply_result['structuredContent']}")

        print("\n👋 Testing greet_user('Alice')...")
        greet_result = call_mcp("tools/call", {"name": "greet_user", "arguments": {"name": "Alice"}})
        print(f"   Result: {greet_result['structuredContent']}")

        print("\n✅ MCP tool testing completed!")

    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        import traceback

        print("\n🔍 Full error traceback:")
        traceback.print_exc()
        sys.exit(1)

if __name__ == "__main__":
    main()


### Testar Invocação de Ferramentas

Vamos testar nossas ferramentas MCP invocando nosso cliente recém-criado:

In [ ]:
print("Testing MCP tool invocation...")
print("=" * 50)
!python invoke_mcp_tools_boto3.py

### Demonstração do Ciclo de Vida da Sessão: Parando Após Teste Boto3

Após testar com a abordagem Boto3, vamos demonstrar como parar outra sessão.
Isso mostra que o gerenciamento de sessão funciona consistentemente entre diferentes métodos de invocação.

In [ ]:
demo3_session_id = str(uuid.uuid4())
print(f"📝 Demo 3 - Generated mcpSessionId: {demo3_session_id}")

async def test3():
    async with streamablehttp_client_with_sigv4(
        url=mcp_url, credentials=credentials, service="bedrock-agentcore",
        region=region, headers={"Mcp-Session-Id": demo3_session_id}
    ) as (r, w, _):
        async with ClientSession(r, w) as s:
            await s.initialize()
            print(f"✅ Session created")

await test3()

print(f"🛑 Stopping session '{demo3_session_id}'...")
response = agentcore_client.stop_runtime_session(
    agentRuntimeArn=launch_result.agent_arn,
    runtimeSessionId=demo3_session_id,
    qualifier='DEFAULT'
)
print(f"✅ Session stopped (HTTP {response['ResponseMetadata']['HTTPStatusCode']})")
print(f"   Request ID: {response['ResponseMetadata']['RequestId']}")
print(f"💡 All demos complete - runtime handled multiple sessions!")

## Próximos Passos

Agora que você implantou com sucesso um servidor MCP no AgentCore Runtime, você pode:

1. **Adicionar Mais Ferramentas**: Estenda seu servidor MCP com ferramentas adicionais
2. **Autenticação Personalizada**: Implemente autenticação de entrada AWS IAM
3. **Integração**: Integre com outros serviços do AgentCore

## Melhores Práticas do Ciclo de Vida da Sessão

Os custos do AgentCore Runtime são baseados em vCPU e Memória. Uma melhor prática para evitar custos indesejados é parar explicitamente a sessão ou configurar um timeout de inatividade apropriado, para que a sessão seja encerrada.

Para gerenciar custos de forma eficaz:

- **Configure timeout de inatividade**: Defina um timeout de inatividade apropriado durante a criação da sessão para parar automaticamente sessões inativas. Escolha um valor baseado no seu caso de uso (por exemplo, menor para desenvolvimento/teste, maior para cargas de trabalho de produção).
- **Pare sessões quando terminar**: Use `stop_runtime_session` para liberar os recursos do microVM para uma sessão específica enquanto mantém o runtime ativo para novas sessões.

## Limpeza

Agora vamos limpar o AgentCore Runtime e recursos associados. Deletamos o runtime primeiro para evitar custos indesejados, depois limpamos recursos de suporte como repositórios ECR.

In [ ]:
# --- Cleanup Resources ---
import boto3

agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
ecr_client = boto3.client('ecr', region_name=region)

# Step 1: Delete Parameter Store parameter first to minimize credential exposure window
try:
    ssm_client.delete_parameter(Name='/mcp_server/runtime_iam/agent_arn')
    print("✅ Parameter Store parameter deleted")
except ssm_client.exceptions.ParameterNotFound:
    print("ℹ️  Parameter Store parameter not found")

# Step 2: Delete the agent runtime to stop incurring costs
# AgentCore Runtime costs are based on vCPU and Memory
try:
    agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result.agent_id,
    )
    print(f"✅ Agent runtime '{launch_result.agent_id}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete agent runtime: {e}")

# Step 3: Delete the ECR repository
try:
    ecr_client.delete_repository(
        repositoryName=launch_result.ecr_uri.split('/')[1],
        force=True
    )
    print(f"✅ ECR repository deleted")
except Exception as e:
    print(f"⚠️ Failed to delete ECR repository: {e}")

print("\n✅ Cleanup completed successfully!")

# Parabéns!

Você completou com sucesso:

✅ **Criou um servidor MCP** com ferramentas personalizadas  
✅ **Testou localmente** usando cliente MCP  
✅ **Configurou autenticação** com Amazon Cognito  
✅ **Implantou na AWS** usando AgentCore Runtime  
✅ **Invocou remotamente** com autenticação adequada  
✅ **Aprendeu conceitos MCP** e melhores práticas  

Seu servidor MCP agora está rodando no Amazon Bedrock AgentCore Runtime e pronto para uso em produção!

## Resumo

Neste tutorial, você aprendeu como:
- Construir servidores MCP usando FastMCP
- Configurar transporte HTTP stateless para compatibilidade com AgentCore
- Configurar autenticação de entrada AWS IAM
- Implantar e gerenciar servidores MCP na AWS
- Testar tanto localmente quanto remotamente
- Usar clientes MCP para invocação de ferramentas

O servidor MCP implantado agora pode ser integrado em aplicações e fluxos de trabalho de IA maiores!